# Manual 3D Fracture ROI Annotation, Replay and Validation

This notebook prepares user-reviewed fracture-region masks for the 13 usable Ruikar knees. It does **not** infer fractures automatically and does **not** add a fifth reconstruction target. The model target remains `[femur, tibia, patella, fibula]`.

## Assumptions locked by the Stage 1 contract

- Annotate on the exact source CT recorded in `preprocessing_metadata.csv`. For Case3 and Case16 this is the cast-cleaned NIfTI, not the raw DICOM.
- Create one binary segment named `fracture_roi` containing the visible fracture gap and immediately adjacent fragments. No automatic dilation is applied.
- Export a binary NIfTI using the source CT as the reference geometry. Do not harden a transform or change the export geometry.
- A verified ROI must survive the locked crop and 200 mm FOV with 100% voxel retention before nearest-neighbour resizing to the 256-cubed LPS grid.
- The user is the sole visual reviewer. Automated PASS means only that geometry and retention checks passed; Agent N approval is still required.

## Manual 3D Slicer workflow

1. Run the setup cells to create or load `reports/manifests/fracture_roi_status_v1.csv`.
2. Load the case's `source_ct_path` in 3D Slicer.
3. Create a segment named `fracture_roi`, annotate the local fracture region, and export it to `source_roi_path` using the source CT reference geometry.
4. Set `roi_status` to `verified_fracture`; otherwise use `no_visible_fracture` or `ungradable`. Record reviewer, date and notes.
5. Run the replay/audit cells. Review every generated three-plane overlay, set `visual_approval` to `approved` or `rejected`, and rerun the audit.

In [6]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage

ROOT = Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "configs").exists(), "run from the project or notebook tree"

CONTRACT = json.loads((ROOT / "configs/data_contract_v1.json").read_text(encoding="utf-8"))
MANIFEST_PATH = ROOT / "reports/manifests/quantitative_manifest_v1.csv"
PREPROCESSING_METADATA_PATH = ROOT / "data/interim/predrr_lps_256_v1/preprocessing_metadata.csv"
STATUS_PATH = ROOT / "reports/manifests/fracture_roi_status_v1.csv"
ROI_ROOT = ROOT / CONTRACT["versioned_outputs"]["fracture_roi"]
SOURCE_ROI_ROOT = ROI_ROOT / "source_original_ct"
QA_ROOT = ROI_ROOT / "qa_v1"
QC_PATH = QA_ROOT / "fracture_roi_qc_v1.csv"
SUMMARY_PATH = QA_ROOT / "fracture_roi_summary_v1.json"
RUN_CONFIG_PATH = QA_ROOT / "run_configuration.json"
HASH_PATH = QA_ROOT / "evidence_sha256.txt"

PENDING_STATUS = "needs_user_review"
FINAL_STATUSES = {"verified_fracture", "no_visible_fracture", "ungradable"}
ALLOWED_STATUSES = FINAL_STATUSES | {PENDING_STATUS}
ALLOWED_APPROVAL = {"pending", "approved", "rejected", "not_applicable"}
BONES = CONTRACT["bone_channels"]
PROCESS_ONLY_IDS = None  # None runs all 13; use a short audited list for a targeted replay.
WRITE_ALIGNED_ROIS = True

print("contract:", CONTRACT["contract_version"])
print("source ROI folder:", SOURCE_ROI_ROOT)
print("aligned ROI folder:", ROI_ROOT)

contract: data_contract_v1
source ROI folder: c:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject\data\interim\fracture_roi_lps_256_v1\source_original_ct
aligned ROI folder: c:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject\data\interim\fracture_roi_lps_256_v1


In [7]:
def reroot_recorded_path(value):
    path = Path(str(value))
    parts = path.parts
    if "data" in parts:
        return ROOT.joinpath(*parts[parts.index("data"):])
    return path


def project_relative(path):
    return Path(path).relative_to(ROOT).as_posix()


manifest = pd.read_csv(MANIFEST_PATH)
dataset_key = manifest["dataset"].astype(str).str.casefold()
fractured = manifest[
    dataset_key.isin({"ruikar", "fractured"}) & (manifest["status"] != "excluded")
].copy().sort_values("sample_id").reset_index(drop=True)
assert len(fractured) == 13, f"expected 13 usable Ruikar knees, found {len(fractured)}"
assert fractured["sample_id"].is_unique

pre_meta = pd.read_csv(PREPROCESSING_METADATA_PATH).set_index("volume_id")
assert set(fractured["sample_id"]) <= set(pre_meta.index), "missing preprocessing metadata"

required_columns = [
    "sample_id", "subject_id", "side", "source_ct_path", "source_roi_path",
    "aligned_roi_path", "roi_status", "reviewer", "review_date",
    "visual_approval", "notes",
]
if STATUS_PATH.exists():
    status = pd.read_csv(STATUS_PATH, keep_default_na=False)
else:
    status = fractured[["sample_id", "subject_id", "side"]].copy()
    status["source_ct_path"] = status["sample_id"].map(
        lambda key: project_relative(reroot_recorded_path(pre_meta.loc[key, "source_path"]))
    )
    status["source_roi_path"] = status["sample_id"].map(
        lambda key: project_relative(SOURCE_ROI_ROOT / f"{key}_fracture_roi_source.nii.gz")
    )
    status["aligned_roi_path"] = status["sample_id"].map(
        lambda key: project_relative(ROI_ROOT / f"{key}_fracture_roi.nii.gz")
    )
    status["roi_status"] = PENDING_STATUS
    status["reviewer"] = ""
    status["review_date"] = ""
    status["visual_approval"] = "pending"
    status["notes"] = ""
    STATUS_PATH.parent.mkdir(parents=True, exist_ok=True)
    status.to_csv(STATUS_PATH, index=False)
    print("wrote review template:", STATUS_PATH)

missing_columns = set(required_columns) - set(status.columns)
assert not missing_columns, f"status manifest missing columns: {sorted(missing_columns)}"
assert len(status) == 13 and status["sample_id"].is_unique
assert set(status["sample_id"]) == set(fractured["sample_id"])
assert set(status["roi_status"]) <= ALLOWED_STATUSES
assert set(status["visual_approval"]) <= ALLOWED_APPROVAL
display(status[required_columns])

,sample_id,subject_id,side,source_ct_path,source_roi_path,aligned_roi_path,roi_status,reviewer,review_date,visual_approval,notes
0,Case11_PartRight,Case11,Right,data/raw/fractured/PartRight/Case11,data/interim/fracture_roi_lps_256_v1/source_or...,data/interim/fracture_roi_lps_256_v1/Case11_Pa...,verified_fracture,author,15-7-2026,pending,
1,Case12_PartRight,Case12,Right,data/raw/fractured/PartRight/Case12,data/interim/fracture_roi_lps_256_v1/source_or...,data/interim/fracture_roi_lps_256_v1/Case12_Pa...,verified_fracture,author,15-7-2026,pending,
2,Case13_PartRight,Case13,Right,data/raw/fractured/PartRight/Case13,data/interim/fracture_roi_lps_256_v1/source_or...,data/interim/fracture_roi_lps_256_v1/Case13_Pa...,verified_fracture,author,15-7-2026,pending,
3,Case14_PartRight,Case14,Right,data/raw/fractured/PartRight/Case14,data/interim/fracture_roi_lps_256_v1/source_or...,data/interim/fracture_roi_lps_256_v1/Case14_Pa...,verified_fracture,author,15-7-2026,pending,
4,Case15_PartRight,Case15,Right,data/raw/fractured/PartRight/Case15,data/interim/fracture_roi_lps_256_v1/source_or...,data/interim/fracture_roi_lps_256_v1/Case15_Pa...,verified_fracture,author,15-7-2026,pending,
5,Case16_PartRight,Case16,Right,data/interim/fractured_cast_cleaned/Case16_cle...,data/interim/fracture_roi_lps_256_v1/source_or...,data/interim/fracture_roi_lps_256_v1/Case16_Pa...,verified_fracture,author,15-7-2026,pending,
6,Case1_PartLeft,Case1,Left,data/raw/fractured/PartLeft/Case1,data/interim/fracture_roi_lps_256_v1/source_or...,data/interim/fracture_roi_lps_256_v1/Case1_Par...,verified_fracture,author,15-7-2026,pending,
7,Case2_PartLeft,Case2,Left,data/raw/fractured/PartLeft/Case2,data/interim/fracture_roi_lps_256_v1/source_or...,data/interim/fracture_roi_lps_256_v1/Case2_Par...,no_visible_fracture,author,15-7-2026,pending,
8,Case3_PartLeft,Case3,Left,data/interim/fractured_cast_cleaned/Case3_clea...,data/interim/fracture_roi_lps_256_v1/source_or...,data/interim/fracture_roi_lps_256_v1/Case3_Par...,verified_fracture,author,15-7-2026,pending,
9,Case5_PartRight,Case5,Right,data/raw/fractured/PartRight/Case5,data/interim/fracture_roi_lps_256_v1/source_or...,data/interim/fracture_roi_lps_256_v1/Case5_Par...,verified_fracture,author,15-7-2026,pending,


In [8]:
TARGET_SIZE = int(CONTRACT["final_shape"][0])
RESAMPLE_SPACING = float(CONTRACT["intermediate_resampling_mm"][0])
FOV_MM = float(CONTRACT["fixed_fov_mm"][0])
FOV_VOXELS = int(round(FOV_MM / RESAMPLE_SPACING))
ORIENTATION = CONTRACT["canonical_orientation"]
HU_MIN, HU_MAX = CONTRACT["intensity_window_hu"]
ROI_INTENSITY_THRESHOLD = 0.1
ROI_CLOSING_RADIUS = 3
ROI_PAD_MARGIN = 5


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().casefold() in {"true", "1", "yes"}


def load_source_ct(key):
    meta = pre_meta.loc[key]
    source = reroot_recorded_path(meta["source_path"])
    assert source.exists(), f"missing recorded source CT for {key}: {source}"
    if meta["source_format"] == "nifti":
        return sitk.ReadImage(str(source))
    reader = sitk.ImageSeriesReader()
    names = reader.GetGDCMSeriesFileNames(str(source))
    assert names, f"no DICOM series for {key}: {source}"
    reader.SetFileNames(names)
    return reader.Execute()


def resample_volume(image, new_spacing=(RESAMPLE_SPACING,) * 3):
    new_size = [
        int(round(size * spacing / target_spacing))
        for size, spacing, target_spacing in zip(image.GetSize(), image.GetSpacing(), new_spacing)
    ]
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputDirection(image.GetDirection())
    resampler.SetOutputOrigin(image.GetOrigin())
    resampler.SetTransform(sitk.Transform())
    resampler.SetDefaultPixelValue(float(sitk.GetArrayViewFromImage(image).min()))
    resampler.SetInterpolator(sitk.sitkLinear)
    return resampler.Execute(image)


def orient_volume(image, orientation=ORIENTATION):
    orienter = sitk.DICOMOrientImageFilter()
    orienter.SetDesiredCoordinateOrientation(orientation)
    return orienter.Execute(image)


def apply_bone_window(array):
    array = np.clip(array, HU_MIN, HU_MAX)
    return ((array - HU_MIN) / (HU_MAX - HU_MIN)).astype(np.float32)


def body_envelope_mask(array, soft_threshold=ROI_INTENSITY_THRESHOLD):
    body = ndimage.binary_opening(
        array > soft_threshold, ndimage.generate_binary_structure(3, 1), iterations=1
    )
    labelled, count = ndimage.label(body)
    if count == 0:
        return array
    sizes = ndimage.sum(np.ones_like(labelled), labelled, range(1, count + 1))
    keep = ndimage.binary_fill_holes(labelled == (np.argmax(sizes) + 1))
    return np.where(keep, array, 0.0).astype(np.float32)


def roi_bone_crop_idx(array):
    mask = (array > ROI_INTENSITY_THRESHOLD).astype(np.uint8)
    structure = ndimage.iterate_structure(
        ndimage.generate_binary_structure(3, 1), ROI_CLOSING_RADIUS
    )
    mask = ndimage.binary_closing(mask, structure=structure).astype(np.uint8)
    labelled, count = ndimage.label(mask)
    if count == 0:
        z1, y1, x1 = array.shape
        return (0, z1, 0, y1, 0, x1)
    sizes = ndimage.sum(mask, labelled, range(1, count + 1))
    coordinates = np.argwhere(labelled == (np.argmax(sizes) + 1))
    lower = np.maximum(0, coordinates.min(0) - ROI_PAD_MARGIN)
    upper = np.minimum(array.shape, coordinates.max(0) + 1 + ROI_PAD_MARGIN)
    return (
        int(lower[0]), int(upper[0]), int(lower[1]),
        int(upper[1]), int(lower[2]), int(upper[2]),
    )


def center_to_fixed_fov(array, size=FOV_VOXELS):
    output = np.zeros((size, size, size), dtype=array.dtype)
    source_slices, destination_slices = [], []
    for length in array.shape:
        if length <= size:
            start = (size - length) // 2
            source_slices.append(slice(0, length))
            destination_slices.append(slice(start, start + length))
        else:
            start = (length - size) // 2
            source_slices.append(slice(start, start + size))
            destination_slices.append(slice(0, size))
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def resize_label(array, target_size=TARGET_SIZE):
    image = sitk.GetImageFromArray(array.astype(np.uint8))
    resampler = sitk.ResampleImageFilter()
    resampler.SetSize([target_size] * 3)
    resampler.SetOutputSpacing([
        image.GetSpacing()[i] * image.GetSize()[i] / target_size for i in range(3)
    ])
    resampler.SetOutputOrigin(image.GetOrigin())
    resampler.SetOutputDirection(image.GetDirection())
    resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    resampler.SetDefaultPixelValue(0)
    resampler.SetTransform(sitk.Transform())
    return (sitk.GetArrayFromImage(resampler.Execute(image)) > 0).astype(np.uint8)


def images_share_grid(left, right, tolerance=1e-5):
    return (
        left.GetSize() == right.GetSize()
        and np.allclose(left.GetSpacing(), right.GetSpacing(), atol=tolerance)
        and np.allclose(left.GetOrigin(), right.GetOrigin(), atol=tolerance)
        and np.allclose(left.GetDirection(), right.GetDirection(), atol=tolerance)
    )


def prepare_case(key):
    native_ct = load_source_ct(key)
    oriented_ct = orient_volume(resample_volume(native_ct))
    ct_array = apply_bone_window(sitk.GetArrayFromImage(oriented_ct).astype(np.float32))
    crop = roi_bone_crop_idx(body_envelope_mask(ct_array))
    return native_ct, oriented_ct, crop, as_bool(pre_meta.loc[key, "si_flipped"])


def replay_roi(source_roi, oriented_ct, crop, si_flip):
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(oriented_ct)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    resampler.SetDefaultPixelValue(0)
    resampler.SetTransform(sitk.Transform())
    roi_oriented = resampler.Execute(sitk.Cast(source_roi > 0, sitk.sitkUInt8))
    pre = (sitk.GetArrayFromImage(roi_oriented) > 0).astype(np.uint8)
    z0, z1, y0, y1, x0, x1 = crop
    cropped = pre[z0:z1, y0:y1, x0:x1]
    boxed = center_to_fixed_fov(cropped)
    aligned = resize_label(boxed)
    if si_flip:
        aligned = np.ascontiguousarray(np.flip(aligned, axis=0))
    return {
        "pre_voxels_0p5mm": int(pre.sum()),
        "crop_voxels_0p5mm": int(cropped.sum()),
        "fov_voxels_0p5mm": int(boxed.sum()),
        "aligned": aligned,
    }


print("locked fracture-ROI replay helpers loaded")

locked fracture-ROI replay helpers loaded


In [9]:
synthetic = np.zeros((11, 9, 7), dtype=np.uint8)
synthetic[4:7, 3:6, 2:5] = 1
boxed = center_to_fixed_fov(synthetic, size=15)
assert boxed.sum() == synthetic.sum(), "center padding must retain every ROI voxel"
clipping_probe = np.ones((11, 9, 7), dtype=np.uint8)
clipped = center_to_fixed_fov(clipping_probe, size=5)
assert clipped.sum() < clipping_probe.sum(), "synthetic clipping guard did not activate"
resized = resize_label(boxed, target_size=8)
assert set(np.unique(resized)) <= {0, 1} and resized.any()

synthetic_image = sitk.GetImageFromArray(synthetic)
full_crop = (0, 11, 0, 9, 0, 7)
full_replay = replay_roi(synthetic_image, synthetic_image, full_crop, si_flip=False)
assert full_replay["pre_voxels_0p5mm"] == full_replay["crop_voxels_0p5mm"]
assert full_replay["pre_voxels_0p5mm"] == full_replay["fov_voxels_0p5mm"]
print("5/5 synthetic replay checks passed")

5/5 synthetic replay checks passed


In [10]:
def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def render_qa(key, ct_array, roi, bone_union, output_path):
    z, y, x = np.round(np.argwhere(roi).mean(axis=0)).astype(int).tolist()
    views = [
        ("Axial", ct_array[z, :, :], roi[z, :, :], bone_union[z, :, :]),
        ("Coronal", ct_array[:, y, :], roi[:, y, :], bone_union[:, y, :]),
        ("Sagittal", ct_array[:, :, x], roi[:, :, x], bone_union[:, :, x]),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for axis, (name, background, roi_slice, bone_slice) in zip(axes, views):
        axis.imshow(background, cmap="gray", origin="lower")
        axis.imshow(
            np.ma.masked_where(roi_slice == 0, roi_slice),
            cmap=ListedColormap(["magenta"]), alpha=0.55, origin="lower",
        )
        if bone_slice.any() and (~bone_slice).any():
            axis.contour(bone_slice.astype(float), levels=[0.5], colors="cyan", linewidths=0.6)
        axis.set_title(f"{name} through ROI centroid")
        axis.axis("off")
    fig.suptitle(f"{key}: magenta=fracture ROI, cyan=four-bone target boundary")
    fig.tight_layout()
    fig.savefig(output_path, dpi=140, bbox_inches="tight")
    plt.close(fig)


requested = set(status["sample_id"]) if PROCESS_ONLY_IDS is None else set(PROCESS_ONLY_IDS)
unknown = requested - set(status["sample_id"])
assert not unknown, f"unknown PROCESS_ONLY_IDS: {sorted(unknown)}"
ROI_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_ROI_ROOT.mkdir(parents=True, exist_ok=True)
QA_ROOT.mkdir(parents=True, exist_ok=True)
run_config = {
    "roi_version": CONTRACT["roi_version"],
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_orientation": "source CT reference geometry",
    "intermediate_spacing_mm": [RESAMPLE_SPACING] * 3,
    "fixed_fov_mm": [FOV_MM] * 3,
    "final_shape": [TARGET_SIZE] * 3,
    "final_spacing_mm": CONTRACT["final_spacing_mm"],
    "required_pre_resize_retention": 1.0,
    "process_only_ids": sorted(requested) if PROCESS_ONLY_IDS is not None else None,
    "writes_aligned_rois": WRITE_ALIGNED_ROIS,
}
RUN_CONFIG_PATH.write_text(json.dumps(run_config, indent=2) + "\n", encoding="utf-8")

manifest_index = manifest.set_index("sample_id")
qc_rows, evidence_files = [], [STATUS_PATH, RUN_CONFIG_PATH]
for row in status.sort_values("sample_id").itertuples(index=False):
    if row.sample_id not in requested:
        continue
    record = {
        "sample_id": row.sample_id,
        "roi_status": row.roi_status,
        "visual_approval": row.visual_approval,
        "result": row.roi_status if row.roi_status != "verified_fracture" else "pending_replay",
        "failure_reason": "",
    }
    if row.roi_status != "verified_fracture":
        qc_rows.append(record)
        continue
    try:
        assert str(row.reviewer).strip(), "reviewer is required for verified_fracture"
        assert str(row.review_date).strip(), "review_date is required for verified_fracture"
        source_roi_path = ROOT / row.source_roi_path
        aligned_roi_path = ROOT / row.aligned_roi_path
        assert source_roi_path.exists(), f"missing source ROI: {source_roi_path}"

        native_ct, oriented_ct, crop, si_flip = prepare_case(row.sample_id)
        source_roi = sitk.ReadImage(str(source_roi_path))
        assert images_share_grid(source_roi, native_ct), (
            "source ROI grid must exactly match the recorded source CT; export from Slicer "
            "with the source CT as reference geometry"
        )
        source_array = sitk.GetArrayFromImage(source_roi)
        assert np.count_nonzero(source_array) > 0, "source ROI is empty"
        assert set(np.unique(source_array)) <= {0, 1}, "source ROI must be binary 0/1"

        replay = replay_roi(source_roi, oriented_ct, crop, si_flip)
        pre_count = replay["pre_voxels_0p5mm"]
        crop_count = replay["crop_voxels_0p5mm"]
        fov_count = replay["fov_voxels_0p5mm"]
        aligned = replay["aligned"]
        assert pre_count > 0, "ROI vanished during 0.5 mm LPS resampling"
        assert crop_count == pre_count, f"ROI crop removed {pre_count - crop_count} voxels"
        assert fov_count == pre_count, f"200 mm FOV removed {pre_count - fov_count} voxels"
        assert aligned.any(), "ROI vanished during 256-cubed nearest-neighbour resize"

        predrr_path = ROOT / manifest_index.loc[row.sample_id, "predrr_path"]
        predrr_image = sitk.ReadImage(str(predrr_path))
        aligned_image = sitk.GetImageFromArray(aligned.astype(np.uint8))
        aligned_image.CopyInformation(predrr_image)
        if WRITE_ALIGNED_ROIS:
            aligned_roi_path.parent.mkdir(parents=True, exist_ok=True)
            sitk.WriteImage(aligned_image, str(aligned_roi_path))
            evidence_files.append(aligned_roi_path)

        target_dir = ROOT / manifest_index.loc[row.sample_id, "target_path"]
        bone_union = np.zeros_like(aligned, dtype=bool)
        for bone in BONES:
            target_image = sitk.ReadImage(str(target_dir / f"{row.sample_id}_{bone}.nii.gz"))
            assert images_share_grid(target_image, predrr_image), f"{bone} target grid mismatch"
            bone_union |= sitk.GetArrayFromImage(target_image) > 0
        bone_intersection = int((aligned.astype(bool) & bone_union).sum())
        assert bone_intersection > 0, "fracture ROI does not intersect any target bone"

        qa_path = QA_ROOT / f"{row.sample_id}_fracture_roi_overlay.png"
        render_qa(
            row.sample_id, sitk.GetArrayFromImage(predrr_image),
            aligned.astype(bool), bone_union, qa_path,
        )
        evidence_files.append(qa_path)
        record.update({
            "result": "pass",
            "source_voxels_native": int(np.count_nonzero(source_array)),
            "pre_voxels_0p5mm": pre_count,
            "crop_voxels_0p5mm": crop_count,
            "fov_voxels_0p5mm": fov_count,
            "pre_resize_retention_fraction": fov_count / pre_count,
            "aligned_voxels_256": int(aligned.sum()),
            "bone_intersection_voxels_256": bone_intersection,
            "si_flipped": si_flip,
        })
    except Exception as exc:
        record["result"] = "fail"
        record["failure_reason"] = str(exc)
    qc_rows.append(record)

qc = pd.DataFrame(qc_rows).sort_values("sample_id")
qc.to_csv(QC_PATH, index=False)
evidence_files.append(QC_PATH)
failure_rows = qc[qc["result"] == "fail"]
pending_rows = status[status["roi_status"] == PENDING_STATUS]
verified_rows = status[status["roi_status"] == "verified_fracture"]
approved_verified = verified_rows[verified_rows["visual_approval"] == "approved"]
rejected_verified = verified_rows[verified_rows["visual_approval"] == "rejected"]
partial_run = PROCESS_ONLY_IDS is not None
if partial_run:
    verdict = "PARTIAL_RUN"
elif not failure_rows.empty or not rejected_verified.empty:
    verdict = "RETRY"
elif not pending_rows.empty:
    verdict = "PENDING_USER_ANNOTATION"
elif len(approved_verified) != len(verified_rows):
    verdict = "PENDING_USER_VISUAL_APPROVAL"
else:
    verdict = "PASS_AUTOMATED_READY_FOR_AGENT_N"
summary = {
    "roi_version": CONTRACT["roi_version"],
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "cohort_rows": int(len(status)),
    "processed_rows": int(len(qc)),
    "status_counts": {str(k): int(v) for k, v in status["roi_status"].value_counts().items()},
    "verified_replay_passes": int((qc["result"] == "pass").sum()),
    "failure_rows": int(len(failure_rows)),
    "approved_verified_rows": int(len(approved_verified)),
    "stage1_fracture_roi_verdict": verdict,
    "agent_n_review_required": True,
    "failures": failure_rows[["sample_id", "failure_reason"]].to_dict("records"),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
evidence_files.append(SUMMARY_PATH)
unique_evidence = sorted(
    {Path(path) for path in evidence_files if Path(path).exists()},
    key=lambda path: path.as_posix(),
)
HASH_PATH.write_text(
    "".join(f"{file_sha256(path)}  {project_relative(path)}\n" for path in unique_evidence),
    encoding="utf-8",
)
display(qc)
print(json.dumps(summary, indent=2))
assert failure_rows.empty, "FRACTURE ROI REPLAY FAILED; inspect the QC table and overlays"
print("FRACTURE ROI AUTOMATED VERDICT:", verdict)

,sample_id,roi_status,visual_approval,result,failure_reason
0,Case11_PartRight,verified_fracture,pending,fail,missing source ROI: c:\Users\Chan Zheng Shao\O...
1,Case12_PartRight,verified_fracture,pending,fail,missing source ROI: c:\Users\Chan Zheng Shao\O...
2,Case13_PartRight,verified_fracture,pending,fail,missing source ROI: c:\Users\Chan Zheng Shao\O...
3,Case14_PartRight,verified_fracture,pending,fail,missing source ROI: c:\Users\Chan Zheng Shao\O...
4,Case15_PartRight,verified_fracture,pending,fail,missing source ROI: c:\Users\Chan Zheng Shao\O...
5,Case16_PartRight,verified_fracture,pending,fail,missing source ROI: c:\Users\Chan Zheng Shao\O...
6,Case1_PartLeft,verified_fracture,pending,fail,missing source ROI: c:\Users\Chan Zheng Shao\O...
7,Case2_PartLeft,no_visible_fracture,pending,no_visible_fracture,
8,Case3_PartLeft,verified_fracture,pending,fail,missing source ROI: c:\Users\Chan Zheng Shao\O...
9,Case5_PartRight,verified_fracture,pending,fail,missing source ROI: c:\Users\Chan Zheng Shao\O...


{
  "roi_version": "fracture_roi_lps_256_v1",
  "completed_at_utc": "2026-07-15T12:49:58.701028+00:00",
  "cohort_rows": 13,
  "processed_rows": 13,
  "status_counts": {
    "verified_fracture": 12,
    "no_visible_fracture": 1
  },
  "verified_replay_passes": 0,
  "failure_rows": 12,
  "approved_verified_rows": 0,
  "stage1_fracture_roi_verdict": "RETRY",
  "agent_n_review_required": true,
  "failures": [
    {
      "sample_id": "Case11_PartRight",
      "failure_reason": "missing source ROI: c:\\Users\\Chan Zheng Shao\\OneDrive\\Desktop\\Github Repo\\TestProject\\TestProject\\data\\interim\\fracture_roi_lps_256_v1\\source_original_ct\\Case11_PartRight_fracture_roi_source.nii.gz"
    },
    {
      "sample_id": "Case12_PartRight",
      "failure_reason": "missing source ROI: c:\\Users\\Chan Zheng Shao\\OneDrive\\Desktop\\Github Repo\\TestProject\\TestProject\\data\\interim\\fracture_roi_lps_256_v1\\source_original_ct\\Case12_PartRight_fracture_roi_source.nii.gz"
    },
    {
      "s

AssertionError: FRACTURE ROI REPLAY FAILED; inspect the QC table and overlays

## HPC handoff and gate

The full 13-case replay may run through Open OnDemand after the manual Slicer exports and status manifest are uploaded. Return the executed notebook or log, edited status CSV, run configuration, QC CSV, summary JSON, all overlays, resource usage, job/output paths and hashes.

A successful automated run must report `PASS_AUTOMATED_READY_FOR_AGENT_N`, 100% pre-resize retention for every verified fracture ROI, zero replay failures, and explicit user approval for every generated overlay. This notebook cannot issue the final Stage 1 PASS.